In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
#from scrapy import fetch_website_links, fetch_website_contents
from openai import OpenAI

from parsing_url import redacted_links, redacted_contents, \
    redacted_images, redacted_headings, redacted_metadata


In [5]:
# Example usage
url = "https://edwarddonner.com"
print("Links:", redacted_links(url)[:5])       # show first 5 links
#print("Contents:", redacted_contents(url)[:5]) # show first 5 paragraphs
#print("Images:", redacted_images(url)[:5])
#print("Headings:", redacted_headings(url)[:5])
#print("Metadata:", redacted_metadata(url))


Links: ['#wp--skip-link--target', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/']


In [6]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = redacted_links(url)
    user_prompt += "\n".join(links)
    return user_prompt



In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'GitHub repository', 'url': 'https://github.com/huggingface'},
  {'type': 'discussion forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'Discord join', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/huggingface'},
  

In [13]:
def fetch_page_and_all_relevant_links(url):
    page_contents = redacted_contents(url)
    relevant_links = select_relevant_links(url)

    result = "### Landing Page Contents\n"
    result += "\n".join(page_contents)  # join list into string

    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        # join list of contents into string
        result += "\n".join(redacted_contents(link["url"]))
    return result


In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

### Landing Page Contents
The platform where the machine learning community collaborates on models, datasets, and applications.
Run a 1-bit 27B LLM locally in your browser on WebGPU
Powerful image editing - supports one or two input images.
Reproduce every ICML 2026 paper with your agent
generate a video from an image with a text prompt
10eros ltx 2.3 image-to-video with native audio
Create, discover and collaborate on ML better.
Host and collaborate on unlimited public models, datasets and applications.
With the HF Open source stack.
Text, image, video, audio or even 3D.
Share your work with the world and build your ML profile.
We provide paid Compute and Enterprise solutions.
Give your team the most advanced platform to build AI with enterprise-grade security, access controls and
			dedicated support.
Starting at $20/user/month
Access 45,000+ models from leading AI providers through a single, unified API with no service fees.
Deploy on optimizedInference Endpointsor update yourSpaces

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n### Landing Page Contents\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nRun a 1-bit 27B LLM locally in your browser on WebGPU\nPowerful image editing - supports one or two input images.\nReproduce every ICML 2026 paper with your agent\ngenerate a video from an image with a text prompt\n10eros ltx 2.3 image-to-video with native audio\nCreate, discover and collaborate on ML better.\nHost and collaborate on unlimited public models, datasets and applications.\nWith the HF Open source stack.\nText, image, video, audio or even 3D.\nShare your work with the world and build your ML profile.\nWe provide paid Compute and Enterprise solutions.\nGive your team the most advanced platform to build AI with enterprise-grade se

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))
    
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The Heartbeat of the Machine Learning Community

---

## About Hugging Face

Hugging Face is a premier collaboration platform designed specifically for the global machine learning community. It serves as a dynamic hub where researchers, engineers, and developers come together to share, explore, and build state-of-the-art machine learning models, datasets, and applications. Positioned at the forefront of AI innovation, Hugging Face empowers users to create and collaborate openly, fostering an ethical AI future.

---

## What We Offer

- **Extensive Model & Dataset Hub**  
  Access over 45,000+ models and datasets spanning text, image, video, audio, and 3D modalities. Our platform supports hosting and collaboration on an unlimited number of public machine learning projects.

- **Cutting-edge Tools & Libraries**  
  Work with state-of-the-art AI and diffusion models in PyTorch, advanced tokenizers, efficient transformers fine-tuning, multi-GPU and TPU training, and novel toolkits such as the Smol Python agent library and TGI for optimized language model serving.

- **Innovative Features**  
  - Run a 1-bit 27B Large Language Model (LLM) directly in your browser on WebGPU.  
  - Powerful image editing supporting one or two input images.  
  - Generate videos from images using text prompts with native audio support.  
  - Reproduce cutting-edge research papers, including upcoming ICML 2026, seamlessly.  

- **Enterprise-Grade Solutions**  
  Designed for organizational scaling with an advanced AI platform that provides:  
  - Enterprise-grade security and granular access controls  
  - Single sign-on (SSO) integration  
  - Comprehensive activity logs and data location auditing  
  - Dedicated support and flexible contract options  
  Pricing starts at $20/user/month for professional compute and collaboration capabilities.

- **Unified API & Deployment Options**  
  Access models from top AI providers via a single API interface with zero service fees. Deploy easily on optimized Inference Endpoints or upgrade Spaces applications to GPU deployments starting at $0.60/hour.

---

## Our Community and Culture

At Hugging Face, collaboration drives creation. We are proud of our fast-growing, vibrant community composed of machine learning engineers, researchers, scientists, and enthusiasts worldwide. Our open-source ethos means we emphasize ethical AI development, transparency, and shared progress for all.

We nurture an inclusive environment that enables learning and innovation, encouraging users to build their ML profiles by sharing work openly and joining the AI revolution. Our talented science and engineering teams work at the cutting edge of technology, constantly pushing the limits of what's possible in machine learning.

---

## Careers at Hugging Face

Join a passionate team devoted to building the foundation of machine learning tooling that is accessible and impactful globally. Whether you are a researcher, engineer, or contributor interested in open source AI, Hugging Face offers exciting career opportunities where your work directly influences the future of AI.

Our culture embraces creativity, continuous learning, and collaboration, providing a space to grow your expertise and work on projects that matter.

---

## Why Choose Hugging Face?

- **Open, collaborative platform trusted by thousands worldwide.**  
- **Access to the latest and most powerful ML models and tools.**  
- **Seamless integration between research, development, deployment, and enterprise needs.**  
- **Strong commitment to ethical AI and community-led innovation.**

---

## Connect and Explore

Discover more, share your work, and join the future of machine learning at:

**Website:** [https://huggingface.co](https://huggingface.co)  
**Brand Assets:** Official logos, colors, and brand materials available for community use.

Unlock the power of AI — together with Hugging Face.

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face - Empowering the Future of Machine Learning

---

## About Hugging Face
Hugging Face is the premier collaboration platform for the global machine learning (ML) community. Serving as a central hub, it enables ML engineers, scientists, and enthusiasts to create, explore, discover, and share open-source machine learning models, datasets, and applications across diverse modalities including text, image, video, audio, and even 3D.

The platform fuels innovation by offering access to over 45,000 cutting-edge AI models from top providers through a unified API — all with no service fees. It supports state-of-the-art research and production-ready tools including:
- PyTorch models and diffusion models
- Fast tokenizers and reinforcement learning pipelines
- Browser-based ML inference and agent building
- Parameter-efficient fine-tuning for large language models
- Multi-GPU, TPU, and mixed precision training

With an open-source stack and an active community, Hugging Face is building the foundational ML tooling required to power the AI revolution and foster an open, ethical AI ecosystem.

---

## Products & Solutions
- **Model & Dataset Hub:** Host and collaborate on unlimited public ML models, datasets, and applications.
- **Compute & Enterprise Solutions:** Scalable cloud infrastructure starting at $20/user/month with enterprise-grade security, single sign-on (SSO), access controls, and dedicated support.
- **Optimized Deployment:** One-click GPU upgrades for Spaces applications and optimized Inference Endpoints priced from $0.60/hour.
- **Unified API:** Seamlessly access thousands of AI models under a single API without additional fees.
- **Advanced Toolkits:** Python clients, TGI toolkit for serving language models, smol library for building agents, and more.

---

## Company Culture
Hugging Face fosters a vibrant, fast-growing community focused on learning, collaboration, and ethical AI development. With a talented science and engineering team at the forefront of technology exploration, the company is deeply committed to open source principles and empowering the next generation of AI innovators to build and share their work freely.

---

## Customers & Use Cases
Hugging Face serves a broad spectrum of customers including:
- Individual AI researchers and developers accelerating ML experimentation
- Startups building intelligent applications powered by foundation models
- Large enterprises requiring secure, scalable AI environments with granular access controls
- Academic institutions reproducing and validating state-of-the-art research papers (e.g., ICML 2026)
- Creative professionals generating multimedia from text prompts (images, video, audio)

---

## Careers & Opportunities
Join Hugging Face to work at the cutting edge of AI innovation within a community-driven and inclusive environment. The company values individuals passionate about open source, ethical AI, and collaborative innovation. Employees contribute to a platform that empowers millions of users globally and helps shape the future of machine learning.

---

## Brand & Identity
Hugging Face’s brand identity centers on collaboration, openness, and community-driven AI development. Official logos, color palettes, and brand assets are available for projects aligned with the company’s mission to democratize machine learning.

---

## Contact & More Information
- Explore models and datasets: https://huggingface.co
- Learn about Enterprise plans and security: https://huggingface.co/enterprise
- Access brand assets: https://huggingface.co/brand

---

**Hugging Face:** The AI platform where the machine learning community builds, shares, and innovates together for an open and ethical AI future.